# Stage 2 proposed system — parser A: fine-tuned 1.5B **selective facts**

Fresh implementation of the **Stage 2 proposed architecture** (no code reused from earlier runs).
The only component that differs from the companion notebook (`02_stage2_proposed_tool_use.ipynb`) is the **intent parser** (section 10) and its configuration.

**This notebook's parser:** A deterministic interpreter reads the request, computes **only** the input statistics it needs (median / mean / mode — full column, one equality group, or one fact per group) and inserts them into the prompt as `Required input facts (sf1)`. The adapter writes the contract in one reply, without tools. If the request needs a statistic the interpreter cannot compute unambiguously (for example a tied mode), the task is withheld before inference (`fact_unavailable`).

**Shared by both notebooks:** contract feedback loop to the parser · Qwen2.5-3B-Instruct FP16 planner/generator · AST safety checker · subprocess sandbox · deterministic verifier · tool-using 3B repair agent (≤ 3 attempts) · identical evaluator and metrics.

```
User request + input CSV
        │
        ▼
┌──────────────────────────────┐   feedback loop (≤ MAX_PARSER_FEEDBACK rounds):
│ INTENT PARSER  (1.5B + LoRA) │◄── schema errors · unknown columns/values ·
│  ← this notebook's variant   │    contract already satisfied by unchanged data ·
└──────────────┬───────────────┘    contract not checkable on this data
               ▼
        Intent contract  ──►  must_change  /  must_preserve
               ▼
┌──────────────────────────────┐
│ PLANNER + GENERATOR (3B fp16)│  plan as comments + def transform(df)
└──────────────┬───────────────┘
               ▼
   Safety checker (AST) ─► Sandbox execute (subprocess, timeout, memory cap)
               ▼
   Before/after profiler ─► Intent verifier (deterministic, 11 predicates)
        ┌──────┼──────────────┐
      PASS  VIOLATION /     UNCERTAIN
        │   crash / unsafe     │
     ACCEPT    ▼            WITHHOLD (flag)
        ┌──────────────────────────────┐
        │ REPAIR AGENT (3B fp16)       │  tools: inspect_rows, column_summary
        │ diagnose → fix → re-execute  │  ≤ MAX_REPAIR_ATTEMPTS, then WITHHOLD
        │ → re-verify                  │
        └──────────────────────────────┘
```

### How to run on Kaggle
1. Settings: **GPU T4 x2**, **Internet on**.
2. Attach the datasets: **`stage2_fresh_data.zip`** (benchmark tasks + 4 CSVs) and `selective_facts_1_5b.zip` (training output; contains `adapter/` with `igt_prompt_version.txt` = `p1-v2+sf1`). Paths are detected automatically.
3. **Run All**. A smoke run (2 dev tasks) runs first, then 20 dev and 100 test tasks. Re-running resumes where it stopped.
4. Download **`/kaggle/working/stage2_fresh_selective_facts.zip`** and compare both runs with `03_compare_parsers.ipynb`.

Outcome per task: **correct accepted** (verifier PASS and output equals the reference) · **incorrect accepted** (silent error) · **withheld** (parser escalation/invalid, verifier UNCERTAIN, or repair budget exhausted).


## 0. Configuration

In [1]:
# ===== Configuration =====
SYSTEM_NAME = "stage2_proposed_selective_facts"
PARSER_PROTOCOL = "p1-v2+sf1"          # adapter is auto-detected by this protocol string
PARSER_BASE = "Qwen/Qwen2.5-1.5B-Instruct"
PARSER_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"   # base revision the adapter was trained on
PARSER_ADAPTER = None                          # None = auto-detect under /kaggle/input; or a path to the adapter/ folder
CODER_BASE = "Qwen/Qwen2.5-3B-Instruct"       # code generator AND repair agent (FP16)

DATA_ROOT = None                               # None = auto-detect (folder containing benchmark/tasks and data/)
INPUT_SEARCH_ROOTS = ["/kaggle/input"]
WORK_DIR = "/kaggle/working"
RUN_NAME = "stage2_fresh_selective_facts"

RUN_MODELS = True
RUN_SMOKE = True                               # 2 dev tasks first, written to RUN_NAME/smoke
SMOKE_TASKS = 2
RUN_SPLITS = ["dev", "test"]                   # 20 dev + 100 test tasks
MAX_TASKS_PER_SPLIT = None                     # e.g. 5 for a quick trial

MAX_PARSER_FEEDBACK = 2                        # parser correction rounds (format / grounding / vacuity)
MAX_REPAIR_ATTEMPTS = 3                        # repair → re-execute → re-verify cycles
MAX_REPAIR_TOOL_CALLS = 3                      # inspection tool calls per repair attempt
PARSER_MAX_NEW_TOKENS = 1024
GEN_MAX_NEW_TOKENS = 1024
REPAIR_MAX_NEW_TOKENS = 1024
EXEC_TIMEOUT_S = 60
REPETITION_PENALTY = 1.0                       # greedy decoding, no repetition penalty (keeps JSON intact)
ATTN_IMPL = "sdpa"                             # use "eager" if your GPU/driver has SDPA problems


## 1. Setup and input discovery
Installs `peft`/`bitsandbytes` if missing and finds the attached datasets automatically.

In [2]:
# ===== Setup: packages, imports, input discovery =====
import os, sys, re, json, math, time, glob, shutil, zipfile, hashlib, difflib, textwrap, traceback, subprocess, pickle, ast
from pathlib import Path
import numpy as np
import pandas as pd

ON_KAGGLE = Path("/kaggle/input").exists()

def _ensure_packages():
    """Install the GPU-side packages only when they are missing (Kaggle images differ between versions)."""
    need = []
    for mod, pip_name in [("peft", "peft"), ("bitsandbytes", "bitsandbytes"), ("accelerate", "accelerate"), ("yaml", "pyyaml")]:
        try:
            __import__(mod)
        except Exception:
            need.append(pip_name)
    if need:
        print("pip install:", need)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

if RUN_MODELS:
    _ensure_packages()
import yaml


def find_input_root(marker_relpath, search_roots):
    """Return the first directory that contains marker_relpath (searched recursively, zips are extracted)."""
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        if (root / marker_relpath).exists():
            return root
        for hit in sorted(root.rglob(Path(marker_relpath).name)):
            cand = hit
            for _ in Path(marker_relpath).parts:
                cand = cand.parent
            if (cand / marker_relpath).exists():
                return cand
        for z in sorted(root.rglob("*.zip")):
            dest = WORK_DIR / "_unzipped" / z.stem
            if not dest.exists():
                try:
                    with zipfile.ZipFile(z) as zf:
                        if not any(n.endswith(Path(marker_relpath).name) for n in zf.namelist()):
                            continue
                        zf.extractall(dest)
                except zipfile.BadZipFile:
                    continue
            found = find_input_root(marker_relpath, [dest])
            if found:
                return found
    return None


def find_adapter(protocol, search_roots):
    """Find an adapter folder whose igt_prompt_version.txt equals the expected protocol string."""
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for ver in sorted(root.rglob("igt_prompt_version.txt")):
            if ver.read_text().strip() == protocol and (ver.parent / "adapter_config.json").exists():
                return ver.parent
        for z in sorted(root.rglob("*.zip")):
            dest = WORK_DIR / "_unzipped" / z.stem
            if not dest.exists():
                try:
                    with zipfile.ZipFile(z) as zf:
                        if not any(n.endswith("adapter/igt_prompt_version.txt") for n in zf.namelist()):
                            continue
                        zf.extractall(dest, members=[n for n in zf.namelist() if n.startswith("adapter/")])
                except zipfile.BadZipFile:
                    continue
            hit = find_adapter(protocol, [dest])
            if hit:
                return hit
    return None


WORK_DIR = Path(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [p for p in INPUT_SEARCH_ROOTS if Path(p).exists()]

DATA_ROOT = Path(DATA_ROOT) if DATA_ROOT else find_input_root("benchmark/tasks/INDEX.md", SEARCH_ROOTS)
assert DATA_ROOT is not None, "Benchmark data not found: attach the stage2_fresh_data dataset (benchmark/tasks + data/*.csv)."
print("DATA_ROOT     :", DATA_ROOT)

if RUN_MODELS:
    PARSER_ADAPTER = Path(PARSER_ADAPTER) if PARSER_ADAPTER else find_adapter(PARSER_PROTOCOL, SEARCH_ROOTS)
    assert PARSER_ADAPTER is not None, f"No adapter with protocol {PARSER_PROTOCOL!r} found under {SEARCH_ROOTS}."
    print("PARSER_ADAPTER:", PARSER_ADAPTER)
    cfg = json.loads((PARSER_ADAPTER / "adapter_config.json").read_text())
    assert cfg["base_model_name_or_path"] == PARSER_BASE, (cfg["base_model_name_or_path"], PARSER_BASE)


pip install: ['bitsandbytes']
DATA_ROOT     : /kaggle/input/datasets/srinivasu2005/stage2-fresh-data
PARSER_ADAPTER: /kaggle/input/datasets/srinivasu2005/sf-adapter-1-5b/adapter


## 2. Benchmark tasks
The system sees only `request` and the input CSV. `reference_code` and `gold_contract` are used **only by the evaluator**.

In [3]:
# ===== Benchmark tasks =====
# Each task: request (what the system sees), data_file (the "before" data),
# reference_code and gold_contract (used ONLY by the evaluator, never by the system).

def load_tasks(data_root):
    tasks = []
    for f in sorted((Path(data_root) / "benchmark" / "tasks").glob("*/*.yaml")):
        t = yaml.safe_load(f.read_text())
        t["request"] = " ".join(str(t["request"]).split())
        tasks.append(t)
    return tasks


_DF_CACHE = {}

def load_before(task):
    """Loading rule of the benchmark: pd.read_csv(path) with no options; index 0..n-1 is the row id."""
    path = Path(DATA_ROOT) / task["data_file"]
    if path not in _DF_CACHE:
        _DF_CACHE[path] = pd.read_csv(path)
    return _DF_CACHE[path].copy()


ALL_TASKS = load_tasks(DATA_ROOT)
print(len(ALL_TASKS), "tasks:", {s: sum(t["split"] == s for t in ALL_TASKS) for s in ("dev", "test")})


120 tasks: {'dev': 20, 'test': 100}


## 3. Intent contract: extraction, schema validation, condition semantics
Written fresh from the contract specification (closed predicate vocabulary, null-safe conditions, value equality with `atol = 1e-9`).

In [4]:
# ===== Intent contract: JSON extraction, schema validation, condition semantics, value equality =====

DTYPES = ("int", "float", "bool", "string", "category", "datetime")
CMP_OPS = ("==", "!=", "<", "<=", ">", ">=")
SET_OPS = ("in", "not_in")
NULL_OPS = ("is_null", "not_null")
COUNT_REL = ("unchanged", "decreased", "increased")
COUNT_ABS = ("eq", "lt", "le", "gt", "ge")
# fn -> argument kinds; "any" = col or lit
FN_SIGNATURES = {
    "identity": [["col"]], "const": [["lit"]], "null": [[]],
    "add": [["any", "any"]], "sub": [["any", "any"]], "mul": [["any", "any"]], "div": [["any", "any"]],
    "round": [["col", "lit"]], "clip": [["col", "lit", "lit"]],
    "lower": [["col"]], "upper": [["col"]], "strip": [["col"]],
    "map": [["col", "lit"]], "cast": [["col", "lit"], ["col", "lit", "lit"]],
}
PRED_FIELDS = {
    "row_count": {"op", "value"},
    "rows_removed_where": {"condition"},
    "rows_retained_where": {"condition"},
    "duplicate_count": {"except", "op", "value", "keep"},
    "null_count": {"op", "value"},
    "column_values_unchanged": {"except", "where"},
    "columns_unchanged": {"except"},
    "column_dtype": {"dtype", "ordered", "categories"},
    "column_value_relation": {"fn", "args", "where", "atol"},
    "columns_added": set(),
    "columns_removed": set(),
}
COMMON_FIELDS = {"type", "target", "id"}


class ContractParseError(Exception):
    pass


def extract_json_objects(text):
    """All top-level JSON objects in a model reply: fenced blocks first, then bare balanced braces."""
    found = []
    for block in re.findall(r"```(?:json|JSON)?\s*(.*?)```", text, flags=re.S):
        try:
            found.append(json.loads(block.strip()))
        except Exception:
            pass
    if found:
        return found
    depth, start, in_str, esc = 0, None, False, False
    for i, ch in enumerate(text):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}" and depth:
            depth -= 1
            if depth == 0:
                try:
                    found.append(json.loads(text[start:i + 1]))
                except Exception:
                    pass
    return found


# ---------------------------------------------------------------- schema validation
def _is_int(v):
    return isinstance(v, int) and not isinstance(v, bool)


def _is_scalar(v):
    return v is None or isinstance(v, (str, int, float, bool))


def validate_condition(cond, path, errors):
    if not isinstance(cond, dict):
        errors.append(f"{path}: condition must be an object"); return
    for bad in ("and", "or"):
        if bad in cond:
            errors.append(f"{path}: there is no '{bad}' key; combine conditions with \"all\"/\"any\"/\"not\"")
            return
    keys = set(cond)
    if keys & {"all", "any"}:
        k = "all" if "all" in cond else "any"
        if keys != {k} or not isinstance(cond[k], list) or not cond[k]:
            errors.append(f"{path}.{k}: must be the only key and a non-empty list of conditions"); return
        for i, c in enumerate(cond[k]):
            validate_condition(c, f"{path}.{k}[{i}]", errors)
        return
    if "not" in keys:
        if keys != {"not"}:
            errors.append(f"{path}.not: must be the only key"); return
        validate_condition(cond["not"], f"{path}.not", errors); return
    if not isinstance(cond.get("column"), str):
        errors.append(f"{path}: leaf condition needs a string \"column\""); return
    op = cond.get("op")
    extra = keys - {"column", "op", "value"}
    if extra:
        errors.append(f"{path}: unknown keys {sorted(extra)} in condition")
    if op in CMP_OPS:
        if "value" not in cond or not _is_scalar(cond["value"]) or cond["value"] is None:
            errors.append(f"{path}: op {op!r} needs a non-null scalar \"value\"")
    elif op in SET_OPS:
        if not isinstance(cond.get("value"), list) or not all(_is_scalar(v) for v in cond["value"]):
            errors.append(f"{path}: op {op!r} needs a list \"value\"")
    elif op in NULL_OPS:
        if "value" in cond:
            errors.append(f"{path}: op {op!r} takes no \"value\"")
    else:
        errors.append(f"{path}: unknown op {op!r}; use one of {list(CMP_OPS + SET_OPS + NULL_OPS)}")


def _validate_count(p, path, errors):
    op = p.get("op")
    if op in COUNT_REL:
        if "value" in p:
            errors.append(f"{path}: relative op {op!r} takes no \"value\"")
    elif op in COUNT_ABS:
        if not _is_int(p.get("value")):
            errors.append(f"{path}: op {op!r} needs an integer \"value\"")
    else:
        errors.append(f"{path}: op must be one of {list(COUNT_REL + COUNT_ABS)}, got {op!r}")


def _validate_cols(target, path, errors, allow_star=True, allow_str=True):
    if target == "*" and allow_star:
        return
    if isinstance(target, str) and allow_str and target != "*":
        return
    if isinstance(target, list) and target and all(isinstance(c, str) for c in target):
        return
    star = '"*", ' if allow_star else ''
    errors.append(f"{path}.target: expected {star}a column name or a list of column names, got {target!r}")


def validate_predicate(p, path, errors):
    if not isinstance(p, dict):
        errors.append(f"{path}: predicate must be an object"); return
    t = p.get("type")
    if t not in PRED_FIELDS:
        errors.append(f"{path}: unknown predicate type {t!r}; use one of {sorted(PRED_FIELDS)}"); return
    path = f"{path}.{t}"
    extra = set(p) - PRED_FIELDS[t] - COMMON_FIELDS
    if extra:
        errors.append(f"{path}: unknown fields {sorted(extra)}")
    tgt = p.get("target")
    if t in ("row_count", "rows_removed_where", "rows_retained_where"):
        if tgt is not None:
            errors.append(f"{path}.target: must be null")
    if t in ("row_count", "null_count", "duplicate_count"):
        _validate_count(p, path, errors)
    if t in ("rows_removed_where", "rows_retained_where"):
        validate_condition(p.get("condition"), f"{path}.condition", errors)
    if t == "duplicate_count":
        _validate_cols(tgt, path, errors, allow_star=True, allow_str=False)
        if "keep" in p:
            if p["keep"] not in ("first", "last"):
                errors.append(f"{path}.keep: must be \"first\" or \"last\"")
            if not (p.get("op") == "eq" and p.get("value") == 0):
                errors.append(f"{path}: \"keep\" appears only with \"op\": \"eq\", \"value\": 0")
    if t == "null_count" and not isinstance(tgt, str):
        errors.append(f"{path}.target: must be one column name")
    if t in ("column_values_unchanged", "columns_unchanged"):
        _validate_cols(tgt, path, errors)
    if "except" in p and not (isinstance(p["except"], list) and all(isinstance(c, str) for c in p["except"])):
        errors.append(f"{path}.except: must be a list of column names")
    if "where" in p and p["where"] is not None:
        validate_condition(p["where"], f"{path}.where", errors)
    if t == "column_dtype":
        if not isinstance(tgt, str):
            errors.append(f"{path}.target: must be one column name")
        if p.get("dtype") not in DTYPES:
            errors.append(f"{path}.dtype: must be one of {list(DTYPES)} (never a pandas name such as int64/object), got {p.get('dtype')!r}")
        if p.get("dtype") != "category" and ("ordered" in p or "categories" in p):
            errors.append(f"{path}: \"ordered\"/\"categories\" appear only with \"dtype\": \"category\"")
        if "ordered" in p and not isinstance(p["ordered"], bool):
            errors.append(f"{path}.ordered: must be true or false")
        if "categories" in p and not isinstance(p["categories"], list):
            errors.append(f"{path}.categories: must be a list")
    if t == "column_value_relation":
        if not isinstance(tgt, str):
            errors.append(f"{path}.target: must be one column name")
        fn = p.get("fn")
        args = p.get("args", [])
        if fn not in FN_SIGNATURES:
            errors.append(f"{path}.fn: unknown function {fn!r}; use one of {sorted(FN_SIGNATURES)} (no mean/median/mode/replace: write the value as a lit)")
            return
        if not isinstance(args, list):
            errors.append(f"{path}.args: must be a list"); return
        kinds = []
        for i, a in enumerate(args):
            if not (isinstance(a, dict) and len(a) == 1 and next(iter(a)) in ("col", "lit")):
                errors.append(f"{path}.args[{i}]: every arg is its own object with exactly one key, {{\"col\": name}} or {{\"lit\": value}}; got {a!r}")
                return
            kinds.append(next(iter(a)))
            if kinds[-1] == "col" and not isinstance(a["col"], str):
                errors.append(f"{path}.args[{i}].col: must be a column name")
        if not any(len(sig) == len(kinds) and all(s in ("any", k) for s, k in zip(sig, kinds)) for sig in FN_SIGNATURES[fn]):
            errors.append(f"{path}: fn {fn} expects args {' or '.join(str(s) for s in FN_SIGNATURES[fn])}, got {kinds}")
            return
        if fn == "round" and not _is_int(args[1]["lit"]):
            errors.append(f"{path}: round needs an integer number of decimals")
        if fn == "map" and not isinstance(args[1]["lit"], dict):
            errors.append(f"{path}: map needs {{\"lit\": {{\"from\": \"to\", ...}}}}")
        if fn == "cast":
            if args[1]["lit"] not in DTYPES:
                errors.append(f"{path}: fn cast: dtype must be one of {list(DTYPES)}")
            if len(args) == 3 and not isinstance(args[2]["lit"], str):
                errors.append(f"{path}: cast format must be a string like \"%d/%m/%Y\"")
        if "atol" in p and not isinstance(p["atol"], (int, float)):
            errors.append(f"{path}.atol: must be a number")
    if t in ("columns_added", "columns_removed"):
        _validate_cols(tgt, path, errors, allow_star=False)


def normalize_contract(obj):
    """Validate a parsed JSON object as an intent contract. Returns (contract, errors)."""
    errors = []
    if not isinstance(obj, dict):
        return None, ["the reply is not a JSON object"]
    if "action" in obj or "tool" in obj:
        return None, ["expected a contract, got a tool call or action"]
    extra = set(obj) - {"contract_version", "must_change", "must_preserve"}
    if extra:
        errors.append(f"unknown top-level keys {sorted(extra)}")
    c = {"contract_version": str(obj.get("contract_version", "1.0")),
         "must_change": obj.get("must_change"), "must_preserve": obj.get("must_preserve")}
    for sec in ("must_change", "must_preserve"):
        if not isinstance(c[sec], list):
            errors.append(f"{sec}: must be a list"); continue
        c[sec] = [dict(p) if isinstance(p, dict) else p for p in c[sec]]
        for i, p in enumerate(c[sec]):
            if isinstance(p, dict):
                if p.get("type") in ("row_count", "rows_removed_where", "rows_retained_where"):
                    p.setdefault("target", None)
                if p.get("type") in ("columns_added", "columns_removed") and isinstance(p.get("target"), str):
                    p["target"] = [p["target"]]
            validate_predicate(p, f"{sec}[{i}]", errors)
    return (c if not errors else None), errors


# ---------------------------------------------------------------- values, dtypes, conditions
def is_null(v):
    if v is None or v is pd.NaT or v is pd.NA:
        return True
    try:
        return bool(pd.isna(v)) if not isinstance(v, (list, dict, tuple, set, np.ndarray)) else False
    except (TypeError, ValueError):
        return False


def _is_number(v):
    return isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, (bool, np.bool_))


def _is_datetime(v):
    return isinstance(v, (pd.Timestamp, np.datetime64)) or type(v).__name__ == "datetime"


def scalar_equal(a, b, atol=1e-9):
    na, nb = is_null(a), is_null(b)
    if na or nb:
        return na and nb
    if _is_number(a) and _is_number(b):
        fa, fb = float(a), float(b)
        if math.isinf(fa) or math.isinf(fb):
            return fa == fb
        return abs(fa - fb) <= atol
    if _is_datetime(a) and _is_datetime(b):
        return pd.Timestamp(a) == pd.Timestamp(b)
    try:
        r = a == b
        return bool(r) if isinstance(r, (bool, np.bool_)) else False
    except Exception:
        return False


def series_equal_mask(x, y, atol=1e-9):
    """Element-wise value equality of two aligned Series (null == null, numbers by value within atol)."""
    xs = x.astype(object) if isinstance(x.dtype, pd.CategoricalDtype) else x
    ys = y.astype(object) if isinstance(y.dtype, pd.CategoricalDtype) else y
    if pd.api.types.is_numeric_dtype(xs) and pd.api.types.is_numeric_dtype(ys) \
            and not pd.api.types.is_bool_dtype(xs) and not pd.api.types.is_bool_dtype(ys):
        xf = pd.to_numeric(xs, errors="coerce").astype(float)
        yf = pd.to_numeric(ys, errors="coerce").astype(float)
        both_null = xf.isna() & yf.isna()
        with np.errstate(invalid="ignore"):
            close = (xf - yf).abs() <= atol
        close |= (xf == yf)  # infinities
        return (both_null | close.fillna(False)).astype(bool)
    return pd.Series([scalar_equal(a, b, atol) for a, b in zip(xs.tolist(), ys.tolist())], index=x.index, dtype=bool)


def dtype_family(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype):
        return "category"
    if pd.api.types.is_bool_dtype(dt):
        return "bool"
    if pd.api.types.is_integer_dtype(dt):
        return "int"
    if pd.api.types.is_float_dtype(dt):
        return "float"
    if pd.api.types.is_datetime64_any_dtype(dt):
        return "datetime" if getattr(dt, "tz", None) is None else "datetime_tz"
    if pd.api.types.is_string_dtype(dt) and not pd.api.types.is_object_dtype(dt):
        return "string"
    if pd.api.types.is_object_dtype(dt):
        vals = s.dropna()
        if all(isinstance(v, str) for v in vals.tolist()):
            return "string"
        return "object"
    return str(dt)


class MissingColumn(Exception):
    def __init__(self, column):
        super().__init__(column)
        self.column = column


def condition_columns(cond, out=None):
    out = set() if out is None else out
    if not isinstance(cond, dict):
        return out
    for k in ("all", "any"):
        if k in cond:
            for c in cond[k]:
                condition_columns(c, out)
            return out
    if "not" in cond:
        return condition_columns(cond["not"], out)
    if "column" in cond:
        out.add(cond["column"])
    return out


def _coerce_value_for(s, v):
    if pd.api.types.is_datetime64_any_dtype(s.dtype) and isinstance(v, str):
        try:
            return pd.Timestamp(v)
        except Exception:
            return v
    return v


def eval_condition(df, cond):
    """Boolean Series over df rows. A null cell makes every comparison false (also != and not_in)."""
    if "all" in cond:
        m = pd.Series(True, index=df.index)
        for c in cond["all"]:
            m &= eval_condition(df, c)
        return m
    if "any" in cond:
        m = pd.Series(False, index=df.index)
        for c in cond["any"]:
            m |= eval_condition(df, c)
        return m
    if "not" in cond:
        return ~eval_condition(df, cond["not"])
    col, op = cond["column"], cond["op"]
    if col not in df.columns:
        raise MissingColumn(col)
    s = df[col]
    if isinstance(s.dtype, pd.CategoricalDtype):
        s = s.astype(object)
    notnull = s.notna()
    if op == "is_null":
        return ~notnull
    if op == "not_null":
        return notnull
    if op in SET_OPS:
        vals = cond["value"]
        hit = pd.Series([any(scalar_equal(x, v, 0) for v in vals) if not is_null(x) else False for x in s.tolist()], index=s.index)
        return (hit if op == "in" else ~hit) & notnull
    v = _coerce_value_for(s, cond["value"])
    def cmp(x):
        if is_null(x):
            return False
        try:
            if op == "==": return scalar_equal(x, v, 0)
            if op == "!=": return not scalar_equal(x, v, 0)
            if op == "<": return x < v
            if op == "<=": return x <= v
            if op == ">": return x > v
            if op == ">=": return x >= v
        except Exception:
            return False
    if pd.api.types.is_numeric_dtype(s) and _is_number(v):
        f = {"==": s.eq, "!=": s.ne, "<": s.lt, "<=": s.le, ">": s.gt, ">=": s.ge}[op]
        return f(v).fillna(False).astype(bool) & notnull
    return pd.Series([bool(cmp(x)) for x in s.tolist()], index=s.index, dtype=bool)


def expand_targets(p, before_cols):
    """Expand a target ("*" + except, a name, or a list) to a list of column names."""
    tgt = p.get("target")
    if tgt == "*":
        exc = set(p.get("except") or [])
        return [c for c in before_cols if c not in exc]
    if isinstance(tgt, str):
        return [tgt]
    return list(tgt or [])


def contract_columns(contract):
    """Columns a contract refers to, split into 'must exist in the input' and 'may be new'."""
    need, new, dtype_targets = set(), set(), set()
    for sec in ("must_change", "must_preserve"):
        for p in contract[sec]:
            t = p["type"]
            for k in ("condition", "where"):
                if p.get(k):
                    need |= condition_columns(p[k])
            if t in ("columns_added",):
                new |= set(p["target"])
            elif t == "column_value_relation":
                new.add(p["target"])
                need |= {a["col"] for a in p.get("args", []) if "col" in a}
            elif t in ("column_dtype",):
                dtype_targets.add(p["target"])
            elif p.get("target") not in (None, "*"):
                need |= set([p["target"]] if isinstance(p["target"], str) else p["target"])
            need |= set(p.get("except") or [])
    need |= dtype_targets - new
    return need, new


## 4. Before/after profiler and deterministic intent verifier
`PASS` → accept, `VIOLATION` → repair, `UNCERTAIN` → flag (withhold).

In [5]:
# ===== Before/after profiler + deterministic intent verifier =====
# Verdict: PASS (every predicate holds) | VIOLATION (some predicate fails -> repair)
#          | UNCERTAIN (the contract itself cannot be checked on this data -> flag, no repair).
# Deviation from the written spec, on purpose: problems caused by the OUTPUT (a required column
# missing, a renumbered index) are VIOLATIONS, because the repair agent can fix them; only problems
# of the CONTRACT (unknown input column, removal condition matching no rows) are UNCERTAIN.

PASS, FAIL, UNCERTAIN = "PASS", "FAIL", "UNCERTAIN"
MAX_EXAMPLES = 5


def _fmt(v):
    if is_null(v):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    if _is_datetime(v):
        return str(pd.Timestamp(v))
    return v if isinstance(v, (str, int, float, bool)) else str(v)


def _count_ok(op, before_n, after_n, value):
    return {"unchanged": after_n == before_n, "decreased": after_n < before_n, "increased": after_n > before_n,
            "eq": after_n == value, "lt": after_n < (value if value is not None else 0), "le": after_n <= (value or 0),
            "gt": after_n > (value or 0), "ge": after_n >= (value or 0)}[op]


# ---------------------------------------------------------------- relation functions
def _arg_series(arg, before_rows):
    if "col" in arg:
        return before_rows[arg["col"]].astype(object) if isinstance(before_rows[arg["col"]].dtype, pd.CategoricalDtype) else before_rows[arg["col"]]
    return pd.Series([arg["lit"]] * len(before_rows), index=before_rows.index, dtype=object)


def _cast_value(v, dtype, fmt=None):
    if is_null(v):
        return None
    try:
        if dtype == "int":
            f = float(v)
            return int(f) if f.is_integer() else None
        if dtype == "float":
            return float(v)
        if dtype == "string":
            return v if isinstance(v, str) else str(_fmt(v))
        if dtype == "bool":
            if isinstance(v, (bool, np.bool_)):
                return bool(v)
            if _is_number(v):
                return {0: False, 1: True}.get(float(v), None)
            return {"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False}.get(str(v).strip().lower())
        if dtype == "category":
            return v
        if dtype == "datetime":
            if _is_datetime(v):
                return pd.Timestamp(v)
            return pd.to_datetime(str(v), format=fmt) if fmt else pd.to_datetime(str(v), format="ISO8601")
    except Exception:
        return None
    return None


def relation_expected(p, before_rows):
    """Expected after-values for column_value_relation, computed from the BEFORE rows."""
    fn, args = p["fn"], p.get("args", [])
    idx = before_rows.index
    if fn == "null":
        return pd.Series([None] * len(idx), index=idx, dtype=object)
    if fn in ("identity", "const"):
        return _arg_series(args[0], before_rows)
    if fn in ("add", "sub", "mul", "div"):
        a, b = _arg_series(args[0], before_rows), _arg_series(args[1], before_rows)
        out = []
        for x, y in zip(a.tolist(), b.tolist()):
            if is_null(x) or is_null(y):
                out.append(None); continue
            try:
                x, y = float(x), float(y)
                out.append({"add": x + y, "sub": x - y, "mul": x * y}.get(fn) if fn != "div" else (None if y == 0 else x / y))
            except Exception:
                out.append(None)
        return pd.Series(out, index=idx, dtype=object)
    src = _arg_series(args[0], before_rows)
    vals = src.tolist()
    if fn == "round":
        n = args[1]["lit"]
        return pd.Series([None if is_null(x) else float(np.round(float(x), n)) for x in vals], index=idx, dtype=object)
    if fn == "clip":
        lo, hi = args[1]["lit"], args[2]["lit"]
        def c(x):
            if is_null(x): return None
            x = float(x)
            if lo is not None: x = max(x, float(lo))
            if hi is not None: x = min(x, float(hi))
            return x
        return pd.Series([c(x) for x in vals], index=idx, dtype=object)
    if fn in ("lower", "upper", "strip"):
        return pd.Series([getattr(x, fn)() if isinstance(x, str) else (None if is_null(x) else x) for x in vals], index=idx, dtype=object)
    if fn == "map":
        m = args[1]["lit"]
        def mp(x):
            if is_null(x): return None
            try:
                if x in m: return m[x]
            except TypeError:
                pass
            key = str(_fmt(x))
            if key in m: return m[key]
            if isinstance(x, (float, np.floating)) and float(x).is_integer() and str(int(x)) in m:
                return m[str(int(x))]
            return x
        return pd.Series([mp(x) for x in vals], index=idx, dtype=object)
    if fn == "cast":
        dtype = args[1]["lit"]
        fmt = args[2]["lit"] if len(args) > 2 else None
        return pd.Series([_cast_value(x, dtype, fmt) for x in vals], index=idx, dtype=object)
    raise ValueError(f"unknown fn {fn}")


# ---------------------------------------------------------------- predicate checks
def _value_mismatch_examples(before, after, rows, col, expected=None, atol=1e-9):
    """Rows (among `rows`) where after[col] != expected (default: before[col])."""
    exp = expected if expected is not None else before.loc[rows, col]
    got = after.loc[rows, col]
    ok = series_equal_mask(got, exp.reindex(rows), atol)
    bad = list(ok.index[~ok.values])
    ex = [{"row": _fmt(r), "before": _fmt(before.at[r, col]) if col in before.columns else None,
           "after": _fmt(got.at[r]), **({"expected": _fmt(exp.at[r])} if expected is not None else {})}
          for r in bad[:MAX_EXAMPLES]]
    return len(bad), ex


def check_predicate(p, before, after, surv):
    """Returns (status, detail, examples)."""
    t = p["type"]
    bcols, acols = set(before.columns), set(after.columns)
    try:
        if t == "row_count":
            ok = _count_ok(p["op"], len(before), len(after), p.get("value"))
            return (PASS if ok else FAIL), f"rows before={len(before)}, after={len(after)}, required op={p['op']}" + (f" {p.get('value')}" if "value" in p else ""), []

        if t in ("rows_removed_where", "rows_retained_where"):
            m = eval_condition(before, p["condition"])
            matched = before.index[m.values]
            if t == "rows_removed_where":
                if len(matched) == 0:
                    return UNCERTAIN, "the removal condition matches no input row", []
                kept = [r for r in matched if r in surv]
                ex = [{"row": _fmt(r), **{c: _fmt(before.at[r, c]) for c in sorted(condition_columns(p["condition"]))}} for r in kept[:MAX_EXAMPLES]]
                return (PASS if not kept else FAIL), f"{len(matched)} input rows match the condition; {len(kept)} of them are still present (all must be removed)", ex
            lost = [r for r in matched if r not in surv]
            ex = [{"row": _fmt(r), **{c: _fmt(before.at[r, c]) for c in sorted(condition_columns(p["condition"]))}} for r in lost[:MAX_EXAMPLES]]
            return (PASS if not lost else FAIL), f"{len(matched)} input rows match the condition; {len(lost)} of them were removed (all must be kept)", ex

        if t == "duplicate_count":
            keys = expand_targets(p, list(before.columns))
            miss = [k for k in keys if k not in acols]
            if miss:
                return FAIL, f"key columns missing from the output: {miss}", []
            n_dup = int(after.duplicated(subset=keys).sum())
            ok = _count_ok(p["op"], int(before.duplicated(subset=keys).sum()), n_dup, p.get("value"))
            detail = f"duplicates on {keys} in output: {n_dup}"
            ex = []
            if ok and "keep" in p:
                expected = set(before.index[~before.duplicated(subset=keys, keep=p["keep"]).values])
                got = set(surv)
                missing = sorted(expected - got)
                wrong = sorted(got - expected)
                if missing or wrong:
                    ok = False
                    detail += f"; with keep={p['keep']}: {len(missing)} rows that should be the surviving {p['keep']} of their group are missing, {len(wrong)} other rows survive"
                    ex = [{"row": _fmt(r), "problem": f"should survive (it is the {p['keep']} of its key group)"} for r in missing[:3]] + \
                         [{"row": _fmt(r), "problem": "should be removed (not the kept row of its key group)"} for r in wrong[:2]]
            return (PASS if ok else FAIL), detail, ex

        if t == "null_count":
            col = p["target"]
            if col not in acols:
                return FAIL, f"column {col!r} is missing from the output", []
            b = int(before[col].isna().sum()) if col in bcols else 0
            a = int(after[col].isna().sum())
            ok = _count_ok(p["op"], b, a, p.get("value"))
            return (PASS if ok else FAIL), f"nulls in {col!r}: before={b}, after={a}, required op={p['op']}" + (f" {p.get('value')}" if "value" in p else ""), []

        if t in ("column_values_unchanged", "columns_unchanged"):
            cols = expand_targets(p, list(before.columns))
            rows = list(surv)
            if p.get("where"):
                wm = eval_condition(before, p["where"])
                wset = set(before.index[wm.values])
                rows = [r for r in rows if r in wset]
            problems, ex = [], []
            for col in cols:
                if col not in bcols:
                    return UNCERTAIN, f"column {col!r} is not in the input data", []
                if col not in acols:
                    problems.append(f"column {col!r} is missing from the output"); continue
                if t == "columns_unchanged":
                    fb, fa = dtype_family(before[col]), dtype_family(after[col])
                    if fb != fa:
                        problems.append(f"column {col!r} changed dtype family {fb} -> {fa}")
                        continue
                n_bad, e = _value_mismatch_examples(before, after, rows, col)
                if n_bad:
                    problems.append(f"column {col!r}: {n_bad} surviving rows changed value")
                    ex += [{"column": col, **x} for x in e[:3]]
            if problems:
                return FAIL, "; ".join(problems[:6]), ex[:MAX_EXAMPLES]
            return PASS, f"{len(cols)} column(s) unchanged on {len(rows)} surviving rows", []

        if t == "column_dtype":
            col = p["target"]
            if col not in acols:
                return FAIL, f"column {col!r} is missing from the output", []
            fam = dtype_family(after[col])
            if fam != p["dtype"]:
                return FAIL, f"column {col!r} has dtype {after[col].dtype} (family {fam}), required {p['dtype']}", []
            if p["dtype"] == "category":
                cat = after[col].cat
                if "ordered" in p and bool(cat.ordered) != p["ordered"]:
                    return FAIL, f"column {col!r} ordered={bool(cat.ordered)}, required ordered={p['ordered']}", []
                if "categories" in p:
                    got = [_fmt(c) for c in cat.categories]
                    want = p["categories"]
                    if p.get("ordered") or cat.ordered:
                        same = len(got) == len(want) and all(scalar_equal(a, b) for a, b in zip(got, want))
                    else:
                        same = len(got) == len(want) and all(any(scalar_equal(a, b) for b in want) for a in got)
                    if not same:
                        return FAIL, f"column {col!r} categories {got}, required {want}", []
            return PASS, f"column {col!r} dtype {after[col].dtype}", []

        if t == "column_value_relation":
            col = p["target"]
            for a in p.get("args", []):
                if "col" in a and a["col"] not in bcols:
                    return UNCERTAIN, f"argument column {a['col']!r} is not in the input data", []
            if col not in acols:
                return FAIL, f"column {col!r} is missing from the output", []
            rows = list(surv)
            if p.get("where"):
                wm = eval_condition(before, p["where"])
                wset = set(before.index[wm.values])
                rows = [r for r in rows if r in wset]
            if not rows:
                return PASS, "no surviving rows match the where-condition", []
            exp = relation_expected(p, before.loc[rows])
            n_bad, ex = _value_mismatch_examples(before, after, rows, col, expected=exp, atol=float(p.get("atol", 1e-9)))
            if n_bad:
                return FAIL, f"{col!r} != {p['fn']}(...) on {n_bad} of {len(rows)} checked rows", ex
            return PASS, f"{col!r} == {p['fn']}(...) on {len(rows)} rows", []

        if t == "columns_added":
            bad = [c for c in p["target"] if c in bcols or c not in acols]
            return (PASS if not bad else FAIL), (f"columns not newly added: {bad}" if bad else f"added {p['target']}"), []

        if t == "columns_removed":
            bad = [c for c in p["target"] if c not in bcols or c in acols]
            return (PASS if not bad else FAIL), (f"columns not removed: {bad}" if bad else f"removed {p['target']}"), []
    except MissingColumn as e:
        return UNCERTAIN, f"condition column {e.column!r} is not in the input data", []
    return UNCERTAIN, f"unsupported predicate {t}", []


def profile_change(before, after):
    """Compact before/after profile used in reports and by the repair agent."""
    if not isinstance(after, pd.DataFrame):
        return {"error": "output is not a DataFrame"}
    surv = after.index.intersection(before.index)
    changed = {}
    for c in before.columns:
        if c in after.columns and len(surv):
            try:
                n = int((~series_equal_mask(after.loc[surv, c], before.loc[surv, c])).sum())
            except Exception:
                n = -1
            if n:
                changed[c] = n
    return {"rows_before": len(before), "rows_after": len(after),
            "rows_removed": int(len(before.index.difference(after.index))),
            "columns_added": [c for c in after.columns if c not in before.columns],
            "columns_removed": [c for c in before.columns if c not in after.columns],
            "dtype_changes": {c: f"{before[c].dtype} -> {after[c].dtype}" for c in before.columns
                              if c in after.columns and str(before[c].dtype) != str(after[c].dtype)},
            "cells_changed_by_column": changed}


def verify(before, after, contract):
    results, structural = [], []
    if not isinstance(after, pd.DataFrame):
        return {"verdict": "VIOLATION", "structural": ["the transformation did not return a DataFrame"], "predicates": []}
    if after.columns.duplicated().any():
        structural.append(f"duplicate column names in the output: {list(after.columns[after.columns.duplicated()])}")
    if after.index.has_duplicates or not after.index.isin(before.index).all():
        structural.append("the output index does not keep the original row ids (do not use reset_index / ignore_index; "
                          "keep the index labels of the input rows)")
    declared_new = set()
    for sec in ("must_change", "must_preserve"):
        for p in contract[sec]:
            if p["type"] == "columns_added":
                declared_new |= set(p["target"])
            if p["type"] in ("column_value_relation", "column_dtype"):
                declared_new.add(p["target"])
    extra = [c for c in after.columns if c not in before.columns and c not in declared_new]
    if extra:
        structural.append(f"the output has columns the contract does not ask for: {extra}")
    if structural:
        return {"verdict": "VIOLATION", "structural": structural, "predicates": [], "profile": profile_change(before, after)}
    surv = set(after.index)
    for sec in ("must_change", "must_preserve"):
        for i, p in enumerate(contract[sec]):
            st, detail, ex = check_predicate(p, before, after, surv)
            results.append({"section": sec, "index": i, "type": p["type"], "status": st, "detail": detail,
                            "examples": ex, "predicate": p})
    stats = [r["status"] for r in results]
    verdict = "VIOLATION" if FAIL in stats else ("UNCERTAIN" if UNCERTAIN in stats else "PASS")
    return {"verdict": verdict, "structural": [], "predicates": results, "profile": profile_change(before, after)}


def render_report(rep, max_items=8):
    """Human/LLM-readable verifier report (failed and uncertain predicates only)."""
    lines = [f"Verifier verdict: {rep['verdict']}"]
    for s in rep.get("structural", []):
        lines.append(f"- STRUCTURE: {s}")
    bad = [r for r in rep.get("predicates", []) if r["status"] != PASS]
    for r in bad[:max_items]:
        lines.append(f"- {r['status']} {r['section']}[{r['index']}] {json.dumps(r['predicate'], default=str)}")
        lines.append(f"    {r['detail']}")
        for e in r["examples"][:3]:
            lines.append(f"    example: {json.dumps(e, default=str)}")
    if rep.get("profile") and "error" not in rep["profile"]:
        lines.append(f"Profile of your output vs input: {json.dumps(rep['profile'], default=str)}")
    return "\n".join(lines)


## 5. Safety checker and sandbox

In [6]:
# ===== Safety checker + sandboxed execution =====
ALLOWED_IMPORTS = {"pandas", "numpy", "re", "math", "datetime", "string"}
FORBIDDEN_NAMES = {"eval", "exec", "compile", "open", "__import__", "input", "globals", "locals", "vars",
                   "getattr", "setattr", "delattr", "breakpoint", "exit", "quit", "help", "memoryview", "os", "sys",
                   "subprocess", "shutil", "socket", "pickle", "importlib", "builtins"}
FORBIDDEN_ATTRS = {"to_csv", "to_pickle", "to_parquet", "to_excel", "to_sql", "to_hdf", "to_feather", "to_json",
                   "to_clipboard", "read_csv", "read_pickle", "read_parquet", "read_excel", "read_sql", "read_json",
                   "read_html", "read_clipboard", "read_table", "read_fwf", "read_hdf", "system", "popen", "remove",
                   "unlink", "rmtree"}


def extract_code(text):
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text, flags=re.S)
    blocks = [b for b in blocks if "def transform" in b] or blocks
    if blocks:
        return blocks[-1].strip()
    if "def transform" in text:
        return text[text.index("def transform"):].strip()
    return None


def safety_check(code):
    """Static AST check. Returns a list of problems (empty = safe to execute)."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return [f"SyntaxError: {e.msg} (line {e.lineno})"]
    problems = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for a in node.names:
                if a.name.split(".")[0] not in ALLOWED_IMPORTS:
                    problems.append(f"import of {a.name!r} is not allowed (only {sorted(ALLOWED_IMPORTS)})")
        elif isinstance(node, ast.ImportFrom):
            if (node.module or "").split(".")[0] not in ALLOWED_IMPORTS:
                problems.append(f"import from {node.module!r} is not allowed")
        elif isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            problems.append(f"use of {node.id!r} is not allowed")
        elif isinstance(node, ast.Attribute):
            if node.attr.startswith("__"):
                problems.append(f"dunder attribute access {node.attr!r} is not allowed")
            elif node.attr in FORBIDDEN_ATTRS:
                problems.append(f"file/system access via .{node.attr} is not allowed")
    if not any(isinstance(n, ast.FunctionDef) and n.name == "transform" for n in tree.body):
        problems.append("the code must define a top-level function transform(df)")
    return sorted(set(problems))


_RUNNER = r'''
import sys, pickle, traceback, json, resource
import numpy as np, pandas as pd
try:
    resource.setrlimit(resource.RLIMIT_AS, (8 * 1024 ** 3, 8 * 1024 ** 3))
except Exception:
    pass
work = sys.argv[1]
df = pd.read_pickle(work + "/input.pkl")
code = open(work + "/code.py").read()
ns = {"pd": pd, "np": np, "__name__": "candidate"}
try:
    exec(compile(code, "<transform>", "exec"), ns)
    out = ns["transform"](df.copy())
    if not isinstance(out, pd.DataFrame):
        raise TypeError(f"transform(df) must return a pandas DataFrame, got {type(out).__name__}")
    out.to_pickle(work + "/output.pkl")
    json.dump({"ok": True}, open(work + "/status.json", "w"))
except Exception as e:
    tb = traceback.format_exc().strip().splitlines()
    json.dump({"ok": False, "error": f"{type(e).__name__}: {e}", "traceback": "\n".join(tb[-8:])}, open(work + "/status.json", "w"))
'''
_RUNNER_PATH = WORK_DIR / "_sandbox_runner.py"
_RUNNER_PATH.write_text(_RUNNER)


def run_in_sandbox(code, before, timeout=EXEC_TIMEOUT_S):
    """Execute transform(df) in a separate Python process (own memory limit + timeout).
    Returns {"ok", "output" (DataFrame|None), "error", "traceback", "seconds"}."""
    work = Path(WORK_DIR) / "_sandbox" / hashlib.sha1(f"{time.time_ns()}{os.getpid()}".encode()).hexdigest()[:12]
    work.mkdir(parents=True, exist_ok=True)
    before.to_pickle(work / "input.pkl")
    (work / "code.py").write_text(code)
    t0 = time.time()
    try:
        env = {**os.environ, "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "CUDA_VISIBLE_DEVICES": ""}
        subprocess.run([sys.executable, str(_RUNNER_PATH), str(work)], timeout=timeout, capture_output=True, cwd=str(work), env=env)
        st = json.loads((work / "status.json").read_text()) if (work / "status.json").exists() else {"ok": False, "error": "sandbox crashed (no status)", "traceback": ""}
    except subprocess.TimeoutExpired:
        st = {"ok": False, "error": f"TimeoutError: execution exceeded {timeout}s", "traceback": ""}
    out = pd.read_pickle(work / "output.pkl") if st.get("ok") else None
    shutil.rmtree(work, ignore_errors=True)
    return {"ok": bool(st.get("ok")), "output": out, "error": st.get("error"), "traceback": st.get("traceback", ""),
            "seconds": round(time.time() - t0, 3)}


def execute_and_verify(code, before, contract):
    """Safety check -> sandbox -> verify. Returns a stage record (without the DataFrame) and the output."""
    rec = {"code": code}
    if code is None:
        rec.update(status="no_code", feedback="Your reply did not contain a ```python block defining transform(df).")
        return rec, None
    problems = safety_check(code)
    if problems:
        rec.update(status="unsafe", safety=problems, feedback="Safety checker rejected the code:\n- " + "\n- ".join(problems))
        return rec, None
    ex = run_in_sandbox(code, before)
    rec["exec_seconds"] = ex["seconds"]
    if not ex["ok"]:
        rec.update(status="exec_error", error=ex["error"], feedback=f"Execution failed: {ex['error']}\n{ex['traceback']}")
        return rec, None
    rep = verify(before, ex["output"], contract)
    rec.update(status=rep["verdict"], report=rep, feedback=render_report(rep))
    return rec, ex["output"]


## 6. Evaluator (answer key)
Compares the final output with the reference output and the predicted contract with the gold contract. Never visible to the system.

In [7]:
# ===== Evaluator (answer key: reference output + gold contract; never visible to the system) =====

def outputs_match(cand, ref, atol=1e-6):
    """Is the candidate output the same table as the reference output?
    Same row ids (index labels), same column set, same dtype family per column, equal values
    (null == null, numbers within atol, categories compared by value + ordered flag + category list)."""
    if not isinstance(cand, pd.DataFrame):
        return False, "candidate is not a DataFrame"
    if set(cand.columns) != set(ref.columns) or cand.columns.duplicated().any():
        return False, f"columns differ: extra={sorted(set(cand.columns) - set(ref.columns))} missing={sorted(set(ref.columns) - set(cand.columns))}"
    if cand.index.has_duplicates or set(cand.index) != set(ref.index):
        return False, f"row ids differ: candidate {len(cand)} rows, reference {len(ref)} rows"
    c = cand.loc[ref.index, list(ref.columns)]
    for col in ref.columns:
        fc, fr = dtype_family(c[col]), dtype_family(ref[col])
        if fc != fr:
            return False, f"column {col!r}: dtype family {fc} vs reference {fr}"
        if fr == "category":
            cc, rc = c[col].cat, ref[col].cat
            if bool(cc.ordered) != bool(rc.ordered):
                return False, f"column {col!r}: ordered flag differs"
            a, b = [_fmt(x) for x in cc.categories], [_fmt(x) for x in rc.categories]
            if (rc.ordered and a != b) or (not rc.ordered and sorted(map(str, a)) != sorted(map(str, b))):
                return False, f"column {col!r}: categories differ"
        eq = series_equal_mask(c[col], ref[col], atol)
        if not eq.all():
            r = eq.index[~eq.values][0]
            return False, f"column {col!r}: {int((~eq).sum())} values differ (row {r}: {_fmt(c.at[r, col])!r} vs {_fmt(ref.at[r, col])!r})"
    return True, "identical"


_REF_CACHE = {}

def reference_output(task):
    if task["id"] not in _REF_CACHE:
        ex = run_in_sandbox(task["reference_code"], load_before(task))
        assert ex["ok"], f"reference failed for {task['id']}: {ex['error']}"
        _REF_CACHE[task["id"]] = ex["output"]
    return _REF_CACHE[task["id"]]


# ---------------------------------------------------------------- contract matching (spec section 5.8)
def _canon_value(v):
    if isinstance(v, bool) or v is None or isinstance(v, str):
        return v
    if isinstance(v, (int, float)):
        return round(float(v), 9)
    if isinstance(v, list):
        return sorted((_canon_value(x) for x in v), key=lambda x: json.dumps(x, sort_keys=True, default=str))
    if isinstance(v, dict):
        return {k: _canon_value(x) for k, x in sorted(v.items())}
    return str(v)


def canon_condition(c):
    if c is None:
        return None
    if "all" in c or "any" in c:
        k = "all" if "all" in c else "any"
        kids = [canon_condition(x) for x in c[k]]
        flat = []
        for x in kids:
            flat += x[k] if isinstance(x, dict) and k in x else [x]
        if len(flat) == 1:
            return flat[0]
        return {k: sorted(flat, key=lambda x: json.dumps(x, sort_keys=True, default=str))}
    if "not" in c:
        inner = canon_condition(c["not"])
        if isinstance(inner, dict) and "not" in inner:
            return inner["not"]
        return {"not": inner}
    out = {"column": c["column"], "op": c["op"]}
    if "value" in c:
        out["value"] = _canon_value(c["value"])
    return out


def predicate_keys(contract, section, before_cols):
    keys = []
    for p in contract.get(section) or []:
        t = p["type"]
        base = {k: v for k, v in p.items() if k not in ("id", "atol", "target", "except", "condition", "where")}
        for k in ("condition", "where"):
            if p.get(k) is not None:
                base[k] = canon_condition(p[k])
        if t in ("columns_unchanged", "column_values_unchanged", "columns_added", "columns_removed"):
            targets = expand_targets(p, before_cols)
        elif t == "duplicate_count":
            targets = [tuple(sorted(expand_targets(p, before_cols)))]
        else:
            targets = [p.get("target")]
        for tg in targets:
            keys.append(json.dumps({"section": section, "target": tg, **_canon_value(base)}, sort_keys=True, default=str))
    return keys


def contract_scores(pred, gold, before_cols):
    out = {}
    for sec in ("must_change", "must_preserve"):
        g = predicate_keys(gold, sec, before_cols)
        p = predicate_keys(pred, sec, before_cols) if pred else []
        gs, ps = set(g), set(p)
        tp = len(gs & ps)
        prec = tp / len(ps) if ps else 0.0
        rec = tp / len(gs) if gs else 1.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        out[sec] = {"precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4), "exact": gs == ps}
    out["exact_match"] = out["must_change"]["exact"] and out["must_preserve"]["exact"]
    return out


## 7. Verifier self-test (CPU)
Runs before any model is loaded: every gold contract must accept its reference output and reject the unchanged input.

In [8]:
# ===== CPU self-test of the verifier on this machine's pandas (no models involved) =====
# Every gold contract must PASS on its reference output and must NOT pass on the unchanged input.
_bad = []
for _t in ALL_TASKS:
    _b = load_before(_t)
    _c, _e = normalize_contract(_t["gold_contract"])
    if _e:
        _bad.append((_t["id"], "gold schema", _e)); continue
    if verify(_b, reference_output(_t), _c)["verdict"] != "PASS":
        _bad.append((_t["id"], "reference rejected", render_report(verify(_b, reference_output(_t), _c))))
    if verify(_b, _b.copy(), _c)["verdict"] == "PASS":
        _bad.append((_t["id"], "no-op accepted", ""))
print(f"verifier self-test: {len(ALL_TASKS) - len(_bad)}/{len(ALL_TASKS)} tasks OK")
for _x in _bad:
    print("  WARNING", _x[0], _x[1], str(_x[2])[:500])


verifier self-test: 120/120 tasks OK


## 8. Parser input format and contract feedback checks (shared by both parser variants)

In [9]:
# ===== Intent parser: shared input format + contract feedback checks =====

def describe_input(df):
    """Exactly the input format both 1.5B adapters were trained on (schema, missing counts, first 3 rows)."""
    lines = [f"Dataset: {len(df)} rows, {df.shape[1]} columns.", "", "Columns (name: dtype, missing values):"]
    lines += [f"- {c}: {df[c].dtype}, {int(df[c].isna().sum())} missing" for c in df.columns]
    return "\n".join(lines) + "\n\nFirst 3 rows (CSV):\n" + df.head(3).to_csv(index=False)


def contract_issues(contract, df):
    """Checks of a schema-valid contract against the INPUT data only (no reference, no gold).
    hard = the contract cannot be used as is; soft = suspicious, worth one round of feedback."""
    hard, soft = [], []
    cols = list(df.columns)
    need, new = contract_columns(contract)
    for c in sorted(need - set(cols)):
        close = difflib.get_close_matches(c, cols, n=3, cutoff=0.5)
        hard.append(f"column {c!r} does not exist in the input data" + (f" (did you mean {close}?)" if close else f"; input columns are {cols}"))
    added = {c for sec in ("must_change", "must_preserve") for p in contract[sec] if p["type"] == "columns_added" for c in p["target"]}
    for c in sorted(new - set(cols) - added):
        soft.append(f"target column {c!r} is not an input column; if a new column is intended, also state it with columns_added, "
                    f"otherwise use the exact input column name")
    if not contract["must_change"]:
        hard.append("must_change is empty; it must state what the request changes")
    # string literals in conditions must be values that occur in the column (exact spelling/case)
    def leaves(cond):
        if not isinstance(cond, dict): return []
        for k in ("all", "any"):
            if k in cond: return [x for c in cond[k] for x in leaves(c)]
        if "not" in cond: return leaves(cond["not"])
        return [cond]
    for sec in ("must_change", "must_preserve"):
        for p in contract[sec]:
            for k in ("condition", "where"):
                for lf in leaves(p.get(k)):
                    col = lf.get("column")
                    if col not in df.columns or df[col].dtype != object:
                        continue
                    vals = lf.get("value")
                    vals = vals if isinstance(vals, list) else ([vals] if isinstance(vals, str) else [])
                    present = set(df[col].dropna().unique().tolist())
                    for v in vals:
                        if isinstance(v, str) and v not in present:
                            sample = sorted(map(str, present))[:15]
                            close = difflib.get_close_matches(v, list(map(str, present)), n=3, cutoff=0.6)
                            soft.append(f"value {v!r} never occurs in column {col!r}" +
                                        (f" (close matches: {close})" if close else f" (values include: {sample})"))
    soft = list(dict.fromkeys(soft))
    if hard:
        return hard, soft
    # the contract must not be satisfied by the unchanged input, and must be checkable on this data
    rep = verify(df, df.copy(), contract)
    if rep["verdict"] == "PASS":
        hard.append("this contract is already satisfied by the unchanged input data, so must_change does not capture "
                    "the requested change (every must_change predicate must be something the correct transformation actually changes)")
    for r in rep["predicates"]:
        if r["status"] == UNCERTAIN:
            hard.append(f"{r['section']}[{r['index']}] {r['type']}: {r['detail']}")
    return list(dict.fromkeys(hard)), soft


def describe_predicate(p):
    """Plain-English rendering of one predicate (used by the planner/generator and the repair agent)."""
    def cond(c):
        if c is None: return "all rows"
        if "all" in c: return "(" + " AND ".join(cond(x) for x in c["all"]) + ")"
        if "any" in c: return "(" + " OR ".join(cond(x) for x in c["any"]) + ")"
        if "not" in c: return f"NOT {cond(c['not'])}"
        if c["op"] in NULL_OPS: return f"{c['column']} {'is missing' if c['op'] == 'is_null' else 'is not missing'}"
        return f"{c['column']} {c['op']} {json.dumps(c.get('value'))}"
    def arg(a): return a["col"] if "col" in a else json.dumps(a["lit"])
    t = p["type"]
    if t == "row_count": return f"row count {p['op']}" + (f" {p['value']}" if "value" in p else "") + " (relative to the input)"
    if t == "rows_removed_where": return f"remove every row where {cond(p['condition'])}"
    if t == "rows_retained_where": return f"keep every row where {cond(p['condition'])}"
    if t == "duplicate_count":
        keys = p["target"] if p["target"] != "*" else f"all columns except {p.get('except', [])}"
        return f"no duplicate rows on key {keys}" + (f"; keep the {p['keep']} row of each key group (original order)" if "keep" in p else "")
    if t == "null_count": return f"missing values in {p['target']}: {p['op']}" + (f" {p['value']}" if "value" in p else "")
    if t == "column_values_unchanged":
        return f"values of {p['target'] if p['target'] != '*' else 'all columns except ' + str(p.get('except', []))} unchanged" + (f" where {cond(p['where'])}" if p.get("where") else "")
    if t == "columns_unchanged":
        return f"columns {p['target'] if p['target'] != '*' else 'all except ' + str(p.get('except', []))} keep the same dtype and values"
    if t == "column_dtype":
        return f"{p['target']} must have dtype {p['dtype']}" + (f", ordered={p['ordered']}" if "ordered" in p else "") + (f", categories={p['categories']}" if "categories" in p else "")
    if t == "column_value_relation":
        return f"{p['target']} = {p['fn']}({', '.join(arg(a) for a in p.get('args', []))})" + (f" where {cond(p['where'])}" if p.get("where") else " on every row")
    if t == "columns_added": return f"add columns {p['target']}"
    if t == "columns_removed": return f"remove columns {p['target']}"
    return json.dumps(p)


def describe_contract(contract):
    lines = ["MUST CHANGE:"] + [f"  - {describe_predicate(p)}" for p in contract["must_change"]]
    lines += ["MUST PRESERVE:"] + [f"  - {describe_predicate(p)}" for p in contract["must_preserve"]]
    return "\n".join(lines)


def tokens_of(calls):
    return sum(c["prompt_tokens"] + c["completion_tokens"] for c in calls)


## 9. Load models
Parser: Qwen2.5-1.5B-Instruct NF4 + the fine-tuned LoRA adapter on GPU 0. Generator **and** repair agent: one Qwen2.5-3B-Instruct FP16 on GPU 1.

In [10]:
# ===== Model loading + chat generation (greedy, token-accounted) =====
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

_TF_NEW = tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 56)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPUs:", torch.cuda.device_count())


def _dtype_kw(dt):
    return {"dtype": dt} if _TF_NEW else {"torch_dtype": dt}


class ChatModel:
    def __init__(self, base_id, device, revision=None, adapter=None, load_4bit=False, name=None):
        self.name = name or base_id
        tok_src = str(adapter) if adapter else base_id
        self.tok = AutoTokenizer.from_pretrained(tok_src, **({} if adapter else {"revision": revision}))
        kw = dict(revision=revision, device_map={"": device}, attn_implementation=ATTN_IMPL, **_dtype_kw(torch.float16))
        if load_4bit:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16,
                                                           bnb_4bit_use_double_quant=False)
        model = AutoModelForCausalLM.from_pretrained(base_id, **kw)
        if adapter:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, str(adapter))
        model.eval()
        self.model = model
        self.device = device
        self.calls = 0

    @torch.inference_mode()
    def chat(self, messages, max_new_tokens=1024, tools=None):
        kw = {"tools": tools} if tools else {}
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kw)
        enc = self.tok(prompt, return_tensors="pt").to(self.device)
        out = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None,
                                  top_p=None, top_k=None, repetition_penalty=REPETITION_PENALTY,
                                  pad_token_id=self.tok.eos_token_id)
        new = out[0, enc["input_ids"].shape[1]:]
        text = self.tok.decode(new, skip_special_tokens=True)
        self.calls += 1
        return text, {"prompt_tokens": int(enc["input_ids"].shape[1]), "completion_tokens": int(new.shape[0]),
                      "truncated": bool(new.shape[0] >= max_new_tokens)}


N_GPU = torch.cuda.device_count()
PARSER_DEVICE = "cuda:0"
CODER_DEVICE = "cuda:1" if N_GPU > 1 else "cuda:0"

t0 = time.time()
PARSER_LM = ChatModel(PARSER_BASE, PARSER_DEVICE, revision=PARSER_REVISION, adapter=PARSER_ADAPTER, load_4bit=True,
                      name=f"{PARSER_BASE}+{PARSER_PROTOCOL}")
CODER_LM = ChatModel(CODER_BASE, CODER_DEVICE, load_4bit=False, name=f"{CODER_BASE} fp16")
print(f"models loaded in {time.time() - t0:.0f}s | parser on {PARSER_DEVICE} (NF4 + adapter) | generator/repair on {CODER_DEVICE} (FP16)")


torch 2.11.0+cu130 | transformers 5.20.2 | GPUs: 2


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

... (7 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

models loaded in 68s | parser on cuda:0 (NF4 + adapter) | generator/repair on cuda:1 (FP16)


## 10. INTENT PARSER — the only cell that differs between the two notebooks

**Variant: `selective_facts` — protocol `p1-v2+sf1`.** A deterministic interpreter reads the request, computes **only** the input statistics it needs (median / mean / mode — full column, one equality group, or one fact per group) and inserts them into the prompt as `Required input facts (sf1)`. The adapter writes the contract in one reply, without tools. If the request needs a statistic the interpreter cannot compute unambiguously (for example a tied mode), the task is withheld before inference (`fact_unavailable`).

The system prompt below is the exact prompt the adapter was fine-tuned with.

In [11]:
# ===== INTENT PARSER (variant A): fine-tuned 1.5B "selective facts" adapter (protocol p1-v2+sf1) =====
# Before the model runs, a deterministic interpreter computes ONLY the input statistics the request needs
# (median / mean / mode, full-column, one equality group, or one fact per group) and inserts them as
# "Required input facts (sf1)". The model then writes the contract in ONE reply. No tools.
# Feedback loop (added): schema errors, unknown columns/values, vacuous or uncheckable contracts are sent
# back to the parser for up to MAX_PARSER_FEEDBACK correction rounds.

PARSER_SYSTEM = 'You turn a data-transformation request into an intent contract: a JSON object that states what the transformation must change and what it must preserve. You never write code.\n\nOutput exactly one JSON object in a ```json block:\n{"contract_version": "1.0", "must_change": [...], "must_preserve": [...]}\n\nmust_change = what the request asks for. must_preserve = everything a reasonable user expects to stay the same: rows not targeted are kept, columns not mentioned are unchanged (values and dtype), values not targeted in mentioned columns are unchanged, no rows are added.\n\nPredicates (use only these; "target" names columns of the input data unless stated):\n- {"type": "row_count", "target": null, "op": OP}\n- {"type": "rows_removed_where", "target": null, "condition": COND}   every input row matching COND is removed\n- {"type": "rows_retained_where", "target": null, "condition": COND}  every input row matching COND is kept\n- {"type": "duplicate_count", "target": [key columns] or "*", "except": [columns], "op": "eq", "value": 0, "keep": "first"|"last"}  one row per key group survives, the first/last in the original order\n- {"type": "null_count", "target": column, "op": OP}\n- {"type": "column_values_unchanged", "target": column | [columns], "where": COND (optional)}  values equal on surviving rows; dtype may change\n- {"type": "columns_unchanged", "target": "*", "except": [columns]}  columns still exist with the same dtype family and values\n- {"type": "column_dtype", "target": column, "dtype": DTYPE, "ordered": true|false (optional), "categories": [...] (optional)}\n- {"type": "column_value_relation", "target": column (may be new), "fn": FN, "args": [ARG, ...], "where": COND (optional)}  after[target] == FN(args of the input row) on surviving rows\n- {"type": "columns_added", "target": [new columns]}\n- {"type": "columns_removed", "target": [columns]}\n\nOP: "unchanged" | "decreased" | "increased" (relative to the input), or "eq" | "lt" | "le" | "gt" | "ge" with "value": integer.\nCOND: {"column": c, "op": "=="|"!="|"<"|"<="|">"|">=", "value": scalar} | {"column": c, "op": "in"|"not_in", "value": [..]} | {"column": c, "op": "is_null"|"not_null"} | {"all": [COND, ...]} | {"any": [COND, ...]} | {"not": COND}. A missing cell makes every comparison false (also "!=" and "not_in"); use "not" when missing cells must be included.\nARG: {"col": column} or {"lit": value}.\nFN: identity [col] | const [lit] | null [] | add, sub, mul, div [a, b] | round [col, lit n] | clip [col, lit lo, lit hi] (null = no bound) | lower, upper, strip [col] | map [col, lit {"from": "to", ...}] (unlisted values unchanged) | cast [col, lit DTYPE].\nDTYPE: "int" | "float" | "bool" | "string" | "category" | "datetime".\n\nUse the exact column names and values shown in the data. Every must_change predicate must be something the correct transformation actually changes.\n\nFormat rules (outputs that break them are rejected):\n- DTYPE is one of the six words int, float, bool, string, category, datetime. Never a pandas name such as int64, float64, object or str.\n- Combine conditions only with "all", "any" and "not". There is no "and" or "or" key.\n- "args" is a list, and every element is its own object with exactly one key: {"col": "name"} or {"lit": value}. Example: clip to at most 60 is "args": [{"col": "hours"}, {"lit": null}, {"lit": 60}].\n- "fn" is one of the FN names above. There is no mean, median, mode or replace; if the request needs a statistic of the data, write its value from the data as a {"lit": ...}.\n- "ordered" and "categories" appear only with "dtype": "category".\n- "keep" appears only together with "op": "eq", "value": 0.\n- Output the JSON block and nothing else.\n\nSelective input facts (sf1): facts are measurements of the INPUT, not transformation instructions.\nUse a fact only for the column, statistic and input-row scope requested. Copy its full-precision value.\nFor missing-value fills, constrain the replacement to originally missing cells (and any requested group), and preserve the other cells.\nUse exact requested column names, deduplication keys and keep rules. A rename uses columns_added, columns_removed and identity; there is no rename function.\nDo not preserve an original column that the request removes or renames. Missing-to-value fills do not increase null counts.\n'


class FactUnavailable(Exception):
    pass


_STATS = [("median", r"\bmedian\b"), ("mean", r"\b(?:mean|average)\b"), ("mode", r"\b(?:mode|most (?:common|frequent))\b")]


def _col_spans(request, cols):
    """(start, column) for every input column mentioned in the request (case-insensitive, whole token)."""
    hits = []
    for c in cols:
        for form in {c, c.replace("_", " ")}:
            for m in re.finditer(r"(?<![\w-])`?" + re.escape(form) + r"`?(?![\w-])", request, flags=re.I):
                hits.append((m.start(), c))
    return sorted(set(hits))


def _match_value(series, raw):
    raw = raw.strip().strip("'\"`")
    vals = series.dropna().unique().tolist()
    for v in vals:
        if str(v) == raw:
            return _fmt(v)
    for v in vals:
        if str(v).lower() == raw.lower():
            return _fmt(v)
    try:
        f = float(raw)
        for v in vals:
            if _is_number(v) and float(v) == f:
                return _fmt(v)
    except ValueError:
        pass
    raise FactUnavailable(f"value {raw!r} not found in column {series.name!r}")


def _statistic(series, stat):
    s = series.dropna()
    if len(s) == 0:
        raise FactUnavailable(f"{series.name!r} has no non-missing values in scope")
    if stat in ("median", "mean"):
        if not pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
            raise FactUnavailable(f"{stat} of non-numeric column {series.name!r}")
        return float(s.median() if stat == "median" else s.mean()), int(len(s))
    vc = s.value_counts()
    top = vc[vc == vc.iloc[0]]
    if len(top) > 1:
        raise FactUnavailable(f"mode of {series.name!r} is tied between {[_fmt(v) for v in top.index[:5]]}")
    return _fmt(top.index[0]), int(len(s))


def compute_facts(request, df):
    found = [s for s, pat in _STATS if re.search(pat, request, flags=re.I)]
    if not found:
        return []
    if len(found) > 1:
        raise FactUnavailable(f"several statistics requested: {found}")
    stat = found[0]
    stat_pat = dict(_STATS)[stat]
    cols = list(df.columns)
    spans = _col_spans(request, cols)
    if not spans:
        raise FactUnavailable("the request names no input column")
    target = None
    for c in sorted(cols, key=len, reverse=True):       # "median of X" / "median X" / "X's median"
        cp = r"`?" + re.escape(c) + r"`?(?![\w-])"
        if re.search(stat_pat + r"\s+(?:value\s+)?(?:of\s+)?(?:the\s+)?(?:column\s+)?" + cp, request, flags=re.I) \
                or re.search(r"(?<![\w-])" + cp + r"'s\s+" + stat_pat, request, flags=re.I):
            target = c
            break
    m_scope = re.search(r"(?:among|within|for|in)\s+(?:the\s+)?rows\s+(?:with|where)\s+`?([\w\- ]+?)`?\s*(?:==|=|is)\s*([^\s,.;]+)", request, flags=re.I)
    scope_col = None
    if m_scope:
        scope_col = next((c for c in cols if c.lower() == m_scope.group(1).strip().lower()), None)
        if scope_col is None:
            raise FactUnavailable(f"scope column {m_scope.group(1)!r} not in the input")
    if target is None:                                     # fall back to the filled column
        m_fill = re.search(r"missing\s+(?:values?\s+(?:in|of)\s+)?`?([\w\-]+)`?", request, flags=re.I)
        cand = [c for _, c in spans if c != scope_col]
        if m_fill and any(c.lower() == m_fill.group(1).lower() for c in cols):
            target = next(c for c in cols if c.lower() == m_fill.group(1).lower())
        elif len(set(cand)) == 1:
            target = cand[0]
        else:
            raise FactUnavailable(f"cannot tell which column the {stat} is of (mentioned: {sorted(set(cand))})")
    full = bool(re.search(r"full[- ]column|whole column|overall|entire column", request, flags=re.I))
    group_col = None
    if not m_scope and not full:
        for m_grp in re.finditer(r"\b(?:same|per|each|by)\b", request, flags=re.I):
            later = [c for pos, c in spans if pos > m_grp.start() and c != target]
            if later:
                group_col = later[0]
                break
    facts = []
    if m_scope:
        val = _match_value(df[scope_col], m_scope.group(2))
        where = {"column": scope_col, "op": "==", "value": val}
        v, n = _statistic(df.loc[df[scope_col] == val, target], stat)
        facts.append({"column": target, "statistic": stat, "value": v, "scope": {"kind": "conditional", "where": where},
                      "nonmissing_count": n, "missing_values": "excluded"})
    elif group_col:
        groups = df[group_col].dropna().unique().tolist()
        if len(groups) > 12:
            raise FactUnavailable(f"too many groups in {group_col!r} ({len(groups)})")
        for g in sorted(groups, key=lambda x: (str(type(x)), x)):
            g = _fmt(g)
            v, n = _statistic(df.loc[df[group_col] == g, target], stat)
            facts.append({"column": target, "statistic": stat, "value": v,
                          "scope": {"kind": "conditional", "where": {"column": group_col, "op": "==", "value": g}},
                          "nonmissing_count": n, "missing_values": "excluded"})
    else:
        v, n = _statistic(df[target], stat)
        facts.append({"column": target, "statistic": stat, "value": v, "scope": {"kind": "full_column"},
                      "nonmissing_count": n, "missing_values": "excluded"})
    return facts


def build_parser_user_message(request, df, facts):
    msg = describe_input(df)
    if facts:
        msg += "\nRequired input facts (sf1):\n" + json.dumps(facts) + "\n"
    return msg + "\nRequest: " + request


def _feedback_message(problems):
    return ("The contract was rejected by the contract checker:\n- " + "\n- ".join(problems) +
            "\nReturn the complete corrected contract as one ```json block and nothing else.")


def run_parser(task, df):
    rec = {"variant": "selective_facts", "calls": [], "feedback": [], "facts": None}
    try:
        facts = compute_facts(task["request"], df)
    except FactUnavailable as e:
        rec.update(status="fact_unavailable", error=str(e))
        return rec
    rec["facts"] = facts
    messages = [{"role": "system", "content": PARSER_SYSTEM},
                {"role": "user", "content": build_parser_user_message(task["request"], df, facts)}]
    contract, errors, hard, soft = None, [], [], []
    for rnd in range(MAX_PARSER_FEEDBACK + 1):
        text, usage = PARSER_LM.chat(messages, max_new_tokens=PARSER_MAX_NEW_TOKENS)
        rec["calls"].append(usage)
        messages.append({"role": "assistant", "content": text})
        objs = [o for o in extract_json_objects(text) if isinstance(o, dict)]
        cands = [o for o in objs if "must_change" in o or "must_preserve" in o]
        if any(o.get("action") == "escalate" for o in objs) and not cands:
            rec.update(status="escalated", error=next(o.get("reason") for o in objs if o.get("action") == "escalate"))
            break
        if not cands:
            contract, errors = None, ["no JSON contract found in the reply" + (" (the reply was cut off; write it more compactly)" if usage["truncated"] else "")]
        else:
            contract, errors = normalize_contract(cands[-1])
        hard, soft = contract_issues(contract, df) if contract else ([], [])
        problems = errors or (hard + soft)
        if not problems:
            break
        rec["feedback"].append({"round": rnd + 1, "problems": problems})
        if rnd == MAX_PARSER_FEEDBACK:
            break
        messages.append({"role": "user", "content": _feedback_message(problems)})
    rec["messages"] = messages
    if rec.get("status") == "escalated":
        return rec
    if errors:
        rec.update(status="parser_invalid", error="; ".join(errors))
    elif hard:
        rec.update(status="parser_rejected", error="; ".join(hard), contract=contract)
    else:
        rec.update(status="ok", contract=contract, warnings=soft)
    return rec


## 11. Transformation planner + code generator (3B)

In [12]:
# ===== Transformation planner + code generator (Qwen2.5-3B-Instruct, FP16) =====
CODE_RULES = """Rules:
1. Define `def transform(df):` that returns a new pandas DataFrame. Start with `out = df.copy()`.
2. `pd` (pandas) and `np` (numpy) are available; you may also import re, math, datetime. No other imports, no files, network, system calls or printing.
3. Keep the original index labels: never call reset_index, never use ignore_index=True, never sort or rebuild the frame. When rows are removed, the remaining rows keep their index labels.
4. Change only what the request asks for. Every other row, column, value and dtype stays exactly as it is. Do not leave helper columns in the output.
5. The output is checked automatically against the intent contract (what must change and what must be preserved). Satisfy every predicate."""

GEN_SYSTEM = "You are a data engineer who writes pandas code. You implement exactly one requested data transformation as a Python function.\n\n" + CODE_RULES


def describe_input_for_coder(df, n_rows=5):
    text = describe_input(df).replace("First 3 rows (CSV):\n" + df.head(3).to_csv(index=False), "")
    text += f"First {n_rows} rows (CSV, index = row id):\n" + df.head(n_rows).to_csv()
    small = {c: sorted(map(str, df[c].dropna().unique()))[:20] for c in df.columns
             if df[c].dtype == object and df[c].nunique() <= 20}
    if small:
        text += "\nValues of low-cardinality text columns:\n" + "\n".join(f"- {c}: {v}" for c, v in small.items())
    return text


def contract_block(contract):
    return describe_contract(contract) + "\n\nContract JSON:\n```json\n" + json.dumps(contract, indent=1, default=str) + "\n```"


def generate_code(task, df, contract):
    user = ("Input data (variable `df`):\n" + describe_input_for_coder(df) +
            f"\n\nRequest: {task['request']}\n\nIntent contract:\n{contract_block(contract)}\n\n"
            "First write a short plan (2-5 bullet points) as Python comments, then the complete function in one ```python block.")
    messages = [{"role": "system", "content": GEN_SYSTEM}, {"role": "user", "content": user}]
    text, usage = CODER_LM.chat(messages, max_new_tokens=GEN_MAX_NEW_TOKENS)
    return {"messages": messages + [{"role": "assistant", "content": text}], "calls": [usage], "code": extract_code(text)}


## 12. Repair agent (3B, tool-using, bounded)

In [13]:
# ===== Repair agent (Qwen2.5-3B-Instruct, FP16) with inspection tools =====
# Bounded loop: diagnose -> (optional tool calls) -> corrected code -> re-execute -> re-verify,
# at most MAX_REPAIR_ATTEMPTS times. The agent never decides acceptance: only the verifier does.

REPAIR_SYSTEM = ("You are the repair agent of a data-transformation system. Code written for a request was executed in a sandbox "
                 "and checked by a deterministic verifier against the intent contract, and it failed. Find the cause and return a "
                 "corrected complete function. You may call the inspection tools first. Your final answer is one short diagnosis "
                 "sentence followed by the complete corrected function in one ```python block.\n\n" + CODE_RULES)

REPAIR_TOOLS = [
    {"type": "function", "function": {
        "name": "inspect_rows",
        "description": "Show the input (before) and current output (after) values of some rows. Removed rows are marked.",
        "parameters": {"type": "object", "properties": {
            "row_ids": {"type": "array", "items": {"type": "integer"}, "description": "up to 10 row ids (index labels)"},
            "columns": {"type": "array", "items": {"type": "string"}, "description": "up to 8 column names"}},
            "required": ["row_ids", "columns"]}}},
    {"type": "function", "function": {
        "name": "column_summary",
        "description": "Compare one column in the input and in the current output: dtype, missing count, distinct values, top values, numeric statistics.",
        "parameters": {"type": "object", "properties": {"column": {"type": "string"}}, "required": ["column"]}}},
]


def _col_summary(s):
    vc = s.dropna().value_counts()
    out = {"dtype": str(s.dtype), "rows": int(len(s)), "missing": int(s.isna().sum()), "distinct": int(len(vc)),
           "top_values": [[_fmt(v), int(n)] for v, n in vc.head(5).items()]}
    if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s) and s.notna().any():
        out.update(min=_fmt(s.min()), max=_fmt(s.max()), mean=float(s.mean()), median=float(s.median()))
    return out


def repair_tool(name, args, before, after):
    try:
        if name == "inspect_rows":
            rows = [r for r in args.get("row_ids", [])][:10]
            cols = [c for c in args.get("columns", [])][:8]
            res = []
            for r in rows:
                item = {"row": r}
                for c in cols:
                    item[f"before.{c}"] = _fmt(before.at[r, c]) if (r in before.index and c in before.columns) else "<no such row/column>"
                    if after is None:
                        item[f"after.{c}"] = "<no output>"
                    elif r not in after.index:
                        item[f"after.{c}"] = "<row removed>"
                    else:
                        item[f"after.{c}"] = _fmt(after.at[r, c]) if c in after.columns else "<column missing>"
                res.append(item)
            return {"ok": True, "rows": res}
        if name == "column_summary":
            c = args.get("column")
            return {"ok": True, "column": c,
                    "before": _col_summary(before[c]) if c in before.columns else "<not in input>",
                    "after": (_col_summary(after[c]) if (after is not None and c in after.columns) else "<not in output>")}
        return {"ok": False, "error": f"unknown tool {name!r}"}
    except Exception as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}"}


def parse_tool_calls(text):
    calls = []
    for blk in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.S):
        try:
            calls.append(json.loads(blk))
        except Exception:
            pass
    if not calls and "```python" not in text:
        calls = [o for o in extract_json_objects(text) if isinstance(o, dict) and "name" in o and "arguments" in o]
    return calls


def repair(task, before, contract, code, first_stage, first_output):
    user = ("Input data (variable `df`):\n" + describe_input_for_coder(before) +
            f"\n\nRequest: {task['request']}\n\nIntent contract:\n{contract_block(contract)}\n\n"
            f"Current code:\n```python\n{code or '# (no code)'}\n```\n\nResult of executing and verifying it:\n{first_stage['feedback']}\n\n"
            f"Diagnose the cause, then return the complete corrected function in one ```python block. "
            f"If you need evidence first, call a tool (at most {MAX_REPAIR_TOOL_CALLS} calls per attempt).")
    messages = [{"role": "system", "content": REPAIR_SYSTEM}, {"role": "user", "content": user}]
    attempts, calls, after = [], [], first_output
    for attempt in range(1, MAX_REPAIR_ATTEMPTS + 1):
        used_tools = []
        while True:
            text, usage = CODER_LM.chat(messages, max_new_tokens=REPAIR_MAX_NEW_TOKENS, tools=REPAIR_TOOLS)
            calls.append(usage)
            messages.append({"role": "assistant", "content": text})
            new_code = extract_code(text)
            tcs = parse_tool_calls(text)
            if new_code is None and tcs and len(used_tools) < MAX_REPAIR_TOOL_CALLS:
                for tc in tcs[: MAX_REPAIR_TOOL_CALLS - len(used_tools)]:
                    res = repair_tool(tc.get("name"), tc.get("arguments") or {}, before, after)
                    used_tools.append({"name": tc.get("name"), "arguments": tc.get("arguments")})
                    messages.append({"role": "tool", "content": json.dumps(res, default=str)})
                continue
            break
        stage, out = execute_and_verify(new_code, before, contract)
        stage["attempt"], stage["tools"] = attempt, used_tools
        attempts.append(stage)
        if out is not None:
            after = out
        if stage["status"] in ("PASS", "UNCERTAIN"):
            return attempts, calls, messages, out
        messages.append({"role": "user", "content": f"Attempt {attempt} was executed and verified again:\n{stage['feedback']}\n\n"
                                                    "Fix the remaining problem and return the complete corrected function in one ```python block."})
    return attempts, calls, messages, out  # output of the last attempt (None if it did not execute)


## 13. Stage 2 pipeline and metrics

In [14]:
# ===== Stage 2 proposed pipeline: parse -> plan+generate -> safety -> sandbox -> verify -> accept | repair | flag =====

def _strip(stage):
    """Stage record without bulky fields (DataFrames are never stored)."""
    s = {k: v for k, v in stage.items() if k != "report"}
    if "report" in stage:
        r = stage["report"]
        s["report"] = {"verdict": r["verdict"], "structural": r.get("structural", []), "profile": r.get("profile"),
                       "failed": [{k: x[k] for k in ("section", "index", "type", "status", "detail", "examples")}
                                  for x in r.get("predicates", []) if x["status"] != "PASS"]}
    return s


def run_task(task):
    t0 = time.time()
    before = load_before(task)
    rec = {k: task[k] for k in ("id", "split", "dataset", "transformation_type", "difficulty", "request")}
    rec["system"] = SYSTEM_NAME
    # 1) intent parser (variant-specific)
    parser = run_parser(task, before)
    rec["parser"] = parser
    rec["tokens"] = {"parser": tokens_of(parser["calls"]), "generator": 0, "repair": 0}
    contract = parser.get("contract") if parser["status"] == "ok" else None
    final_output, rec["stages"] = None, []
    if contract is None:
        rec["decision"], rec["withhold_reason"] = "WITHHOLD", parser["status"]
    else:
        # 2) planner + generator, 3) safety, 4) sandbox, 5) profile + verify
        gen = generate_code(task, before, contract)
        rec["tokens"]["generator"] = tokens_of(gen["calls"])
        rec["generation"] = {"messages": gen["messages"], "calls": gen["calls"]}
        stage, out = execute_and_verify(gen["code"], before, contract)
        stage["attempt"] = 0
        rec["stages"].append(_strip(stage))
        initial_output, final_output = out, out
        # 6) repair loop on VIOLATION / crash / unsafe / no code; UNCERTAIN is flagged, not repaired
        if stage["status"] not in ("PASS", "UNCERTAIN") and MAX_REPAIR_ATTEMPTS > 0:
            attempts, calls, msgs, out = repair(task, before, contract, gen["code"], stage, out)
            rec["stages"] += [_strip(a) for a in attempts]
            rec["tokens"]["repair"] = tokens_of(calls)
            rec["repair"] = {"messages": msgs, "calls": calls}
            final_output = out
        last = rec["stages"][-1]["status"]
        rec["decision"] = "ACCEPT" if last == "PASS" else "WITHHOLD"
        rec["withhold_reason"] = None if last == "PASS" else ("verify_uncertain" if last == "UNCERTAIN" else
                                                              ("repair_exhausted" if len(rec["stages"]) > 1 else f"initial_{last}"))
        # evaluator only: was the initial / final candidate actually correct?
        ref = reference_output(task)
        rec["initial_correct"], rec["initial_match_detail"] = outputs_match(initial_output, ref) if initial_output is not None else (False, "no output")
        rec["contract_scores"] = contract_scores(contract, normalize_contract(task["gold_contract"])[0], list(before.columns))
    ref = reference_output(task)
    rec["final_correct"], rec["final_match_detail"] = outputs_match(final_output, ref) if final_output is not None else (False, "no output")
    if rec["decision"] == "ACCEPT":
        rec["outcome"] = "correct_accepted" if rec["final_correct"] else "incorrect_accepted"
    else:
        rec["outcome"] = "withheld"
    rec["tokens"]["total"] = sum(rec["tokens"].values())
    rec["seconds"] = round(time.time() - t0, 1)
    return rec


def summarize(records):
    n = len(records)
    if not n:
        return {}
    cnt = lambda f: sum(1 for r in records if f(r))
    with_contract = [r for r in records if r.get("contract_scores")]
    mean = lambda xs: round(float(np.mean(xs)), 4) if xs else None
    s = {
        "tasks": n,
        "correct_accepted": cnt(lambda r: r["outcome"] == "correct_accepted"),
        "incorrect_accepted": cnt(lambda r: r["outcome"] == "incorrect_accepted"),
        "withheld": cnt(lambda r: r["outcome"] == "withheld"),
        "withheld_but_final_candidate_correct": cnt(lambda r: r["outcome"] == "withheld" and r["final_correct"]),
        "withhold_reasons": dict(pd.Series([r["withhold_reason"] for r in records if r["withhold_reason"]]).value_counts().to_dict()),
        "parser": {
            "ok": cnt(lambda r: r["parser"]["status"] == "ok"),
            "used_feedback": cnt(lambda r: len(r["parser"]["feedback"]) > 0),
            "recovered_by_feedback": cnt(lambda r: r["parser"]["status"] == "ok" and len(r["parser"]["feedback"]) > 0),
            "feedback_rounds_total": sum(len(r["parser"]["feedback"]) for r in records),
            "tool_attempts": sum(r["parser"].get("tool_attempts", 0) for r in records),
            "tool_failures": sum(r["parser"].get("tool_failures", 0) for r in records),
            "tasks_with_facts": cnt(lambda r: bool(r["parser"].get("facts"))),
            "contract_exact_match": cnt(lambda r: r.get("contract_scores", {}).get("exact_match")),
            "must_change_f1_mean": mean([r["contract_scores"]["must_change"]["f1"] for r in with_contract]),
            "must_preserve_f1_mean": mean([r["contract_scores"]["must_preserve"]["f1"] for r in with_contract]),
        },
        "initial_pass": cnt(lambda r: r["stages"] and r["stages"][0]["status"] == "PASS"),
        "initial_correct": cnt(lambda r: r.get("initial_correct")),
        "entered_repair": cnt(lambda r: len(r["stages"]) > 1),
        "fixed_by_repair": cnt(lambda r: len(r["stages"]) > 1 and r["decision"] == "ACCEPT"),
        "repair_attempts_total": sum(max(0, len(r["stages"]) - 1) for r in records),
        "tokens": {k: int(sum(r["tokens"][k] for r in records)) for k in ("parser", "generator", "repair", "total")},
        "seconds": round(sum(r["seconds"] for r in records), 1),
    }
    s["silent_error_rate_among_accepted"] = round(s["incorrect_accepted"] / max(1, s["correct_accepted"] + s["incorrect_accepted"]), 4)
    return s


def task_row(r):
    p = r["parser"]
    return {"id": r["id"], "split": r["split"], "type": r["transformation_type"], "parser_status": p["status"],
            "parser_feedback_rounds": len(p["feedback"]), "parser_tools": p.get("tool_attempts", 0),
            "facts": len(p.get("facts") or []),
            "contract_exact": (r.get("contract_scores") or {}).get("exact_match"),
            "initial_verdict": r["stages"][0]["status"] if r["stages"] else None,
            "repair_attempts": max(0, len(r["stages"]) - 1), "decision": r["decision"],
            "withhold_reason": r["withhold_reason"], "final_correct": r["final_correct"], "outcome": r["outcome"],
            "tokens": r["tokens"]["total"], "seconds": r["seconds"]}


## 14. Run: smoke → dev (20) → test (100)
Resumable: each finished task is saved as JSON and skipped if the cell is re-run.

In [15]:
# ===== Run: smoke -> dev -> test (resumable; one JSON per task) =====
OUT_DIR = WORK_DIR / RUN_NAME
(OUT_DIR / "tasks").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "config.json").write_text(json.dumps({
    "system": SYSTEM_NAME, "parser_protocol": PARSER_PROTOCOL, "parser_base": PARSER_BASE, "parser_revision": PARSER_REVISION,
    "parser_adapter": str(PARSER_ADAPTER), "coder": CODER_BASE, "max_parser_feedback": MAX_PARSER_FEEDBACK,
    "max_repair_attempts": MAX_REPAIR_ATTEMPTS, "max_repair_tool_calls": MAX_REPAIR_TOOL_CALLS,
    "repetition_penalty": REPETITION_PENALTY, "torch": torch.__version__, "transformers": transformers.__version__,
    "pandas": pd.__version__, "gpus": [torch.cuda.get_device_name(i) for i in range(N_GPU)]}, indent=1))


def run_split(tasks, out_dir, label):
    out_dir.mkdir(parents=True, exist_ok=True)
    records = []
    for i, t in enumerate(tasks, 1):
        f = out_dir / f"{t['id']}.json"
        if f.exists():
            records.append(json.loads(f.read_text()))
            continue
        try:
            r = run_task(t)
        except Exception as e:  # keep going; the failure is recorded as a withheld task
            r = {"id": t["id"], "split": t["split"], "transformation_type": t["transformation_type"], "request": t["request"],
                 "parser": {"status": "pipeline_error", "feedback": [], "calls": []}, "stages": [], "decision": "WITHHOLD",
                 "withhold_reason": "pipeline_error", "error": traceback.format_exc(), "final_correct": False,
                 "outcome": "withheld", "tokens": {"parser": 0, "generator": 0, "repair": 0, "total": 0}, "seconds": 0}
            print(traceback.format_exc())
        f.write_text(json.dumps(r, default=str, indent=1))
        records.append(r)
        row = task_row(r)
        print(f"[{label} {i:3d}/{len(tasks)}] {r['id']:<20} parser={row['parser_status']}(fb={row['parser_feedback_rounds']},"
              f"tools={row['parser_tools']},facts={row['facts']}) initial={row['initial_verdict']} repairs={row['repair_attempts']} "
              f"-> {r['decision']:<8} {r['outcome']:<18} {row['tokens']:>6} tok {row['seconds']:>5}s", flush=True)
    return records


all_records = {}
if RUN_SMOKE:
    smoke = [t for t in ALL_TASKS if t["split"] == "dev"][:SMOKE_TASKS]
    recs = run_split(smoke, OUT_DIR / "smoke", "smoke")
    assert all(r["withhold_reason"] != "pipeline_error" for r in recs), "smoke run hit a pipeline error - see the traceback above"
    print("smoke OK")
for split in RUN_SPLITS:
    sel = [t for t in ALL_TASKS if t["split"] == split][:MAX_TASKS_PER_SPLIT]
    all_records[split] = run_split(sel, OUT_DIR / "tasks", split)

summary = {split: summarize(recs) for split, recs in all_records.items()}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
results = pd.DataFrame([task_row(r) for recs in all_records.values() for r in recs])
results.to_csv(OUT_DIR / "results.csv", index=False)
for split, s in summary.items():
    print(f"\n===== {SYSTEM_NAME} | {split}: {s['tasks']} tasks =====")
    print(f"correct accepted   : {s['correct_accepted']}")
    print(f"incorrect accepted : {s['incorrect_accepted']}   (silent errors)")
    print(f"withheld           : {s['withheld']}   (of which final candidate was actually correct: {s['withheld_but_final_candidate_correct']})")
    print(f"withhold reasons   : {s['withhold_reasons']}")
    print(f"parser             : {s['parser']}")
    print(f"initial PASS {s['initial_pass']} | entered repair {s['entered_repair']} | fixed by repair {s['fixed_by_repair']}")
    print(f"tokens             : {s['tokens']}")


[smoke   1/2] adult_col_006        parser=parser_rejected(fb=3,tools=0,facts=0) initial=None repairs=0 -> WITHHOLD withheld             5727 tok  43.0s
[smoke   2/2] adult_conv_006       parser=ok(fb=0,tools=0,facts=0) initial=PASS repairs=0 -> ACCEPT   correct_accepted     3122 tok  18.0s
smoke OK
[dev   1/20] adult_col_006        parser=parser_rejected(fb=3,tools=0,facts=0) initial=None repairs=0 -> WITHHOLD withheld             5727 tok  41.3s
[dev   2/20] adult_conv_006       parser=ok(fb=0,tools=0,facts=0) initial=PASS repairs=0 -> ACCEPT   correct_accepted     3122 tok  18.2s
[dev   3/20] adult_dedup_006      parser=ok(fb=0,tools=0,facts=0) initial=PASS repairs=0 -> ACCEPT   correct_accepted     3071 tok  15.7s
[dev   4/20] adult_filt_006       parser=ok(fb=0,tools=0,facts=0) initial=PASS repairs=0 -> ACCEPT   correct_accepted     3336 tok  26.4s
[dev   5/20] adult_miss_006       parser=ok(fb=0,tools=0,facts=0) initial=PASS repairs=0 -> ACCEPT   incorrect_accepted   3229 tok  22.

## 15. Package results

In [16]:
# ===== Package the run (download this ZIP from the Output tab) =====
zip_path = WORK_DIR / f"{RUN_NAME}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob("*")):
        if f.is_file():
            zf.write(f, f.relative_to(OUT_DIR.parent))
print("wrote", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")
results.groupby(["split", "outcome"]).size().unstack(fill_value=0)


wrote /kaggle/working/stage2_fresh_selective_facts.zip (0.6 MB)


outcome,correct_accepted,incorrect_accepted,withheld
split,,,
dev,11,1,8
test,71,2,27
